![SGSSS Logo](https://github.com/SGSSSonline/text-analysis/blob/main/img/SGSSS_Stacked.png?raw=true)

# Practical 2: Descriptive Text Analysis

In the previous practical, we learned how to transform raw text into a clean, structured format. In this practical, we put that preprocessed text to work. We explore a range of descriptive text analysis techniques that allow us to characterise the content and patterns within a corpus — from simple word counts to more sophisticated measures of association and comparison.

These methods are foundational: they are often the first step in any text analysis project, helping researchers understand what is in their data before moving on to more complex modelling techniques.

## Aims

This practical has two aims:

1. **Demonstrate how to use Python to perform descriptive text analysis** on social science data.
2. **Cultivate computational thinking skills** — learning to summarise, visualise, and compare textual data using a range of techniques.

### Lesson Details

| | |
| --- | --- |
| **Level** | Introductory |
| **Time** | ~45 minutes |
| **Pre-requisites** | Practical 1: Preprocessing Text |
| **Learning outcomes** | Compute and visualise word frequencies |
| | Generate and interpret word clouds, including comparative word clouds |
| | Use Keywords-in-Context (KWIC) to examine how terms are used |
| | Identify collocations and interpret co-occurrence patterns |
| | Compare vocabulary across groups within a corpus |

## Guide to Using This Resource

This is a **Jupyter Notebook** — an interactive document that combines text, code, and output in a single environment. If you are viewing this in **Google Colab**, you are running the notebook in the cloud, which means you do not need to install anything on your own machine.

A notebook is made up of **cells**. There are two main types:

- **Markdown cells** contain formatted text (like this one). They provide explanations, instructions, and context.
- **Code cells** contain Python code that you can execute. Code cells are displayed with a grey background and have a play button on the left.

To **run a cell**, click on it and press `Shift+Enter` (or click the play button). The output will appear directly below the cell. You should run the code cells **in order**, from top to bottom, as later cells often depend on variables or modules defined in earlier cells.

If you are using **Google Colab**, make sure to save a copy of this notebook to your own Google Drive first: go to **File > Save a copy in Drive**.

If you are new to Jupyter Notebooks and would like a more detailed introduction, see the excellent materials by Dani Arribas-Bel: [https://github.com/darribas/gds19/blob/master/content/labs/lab_00.ipynb](https://github.com/darribas/gds19/blob/master/content/labs/lab_00.ipynb)

### Interactive Test

Let's make sure everything is working. Run the cell below and enter your name when prompted.

In [ ]:
print("Hello! What is your name?")
name = input()
print(f"Welcome to Practical 2, {name}!")

## Setup

Before we begin, we need to install and import the Python libraries we will use throughout this practical. These include:

- **NLTK** — for tokenisation, stopword removal, concordancing, and collocations.
- **wordcloud** — for generating word cloud visualisations.
- **matplotlib** — for plotting charts and displaying images.
- **pandas** — for working with tabular data.
- **Counter** (from collections) — for counting word frequencies.

In [ ]:
# Install packages (if needed on Colab)
!pip install nltk wordcloud matplotlib -q

import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.collocations import BigramCollocationFinder
from nltk import FreqDist, Text
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter

print("Successfully imported necessary modules")

## Load and Preprocess the Corpus

We will work with the same sample of UK parliamentary speeches drawn from the **Hansard debates** that we used in Practical 1. The dataset contains 20 speech excerpts with metadata including the speaker's name, party affiliation, and the date of the speech.

First, let's recreate the dataset.

In [ ]:
# Create the sample dataset of parliamentary speech excerpts
speeches_data = {
    "date": [
        "2024-01-15", "2024-01-15", "2024-01-22", "2024-01-22", "2024-02-05",
        "2024-02-05", "2024-02-12", "2024-02-12", "2024-02-19", "2024-02-19",
        "2024-03-04", "2024-03-04", "2024-03-11", "2024-03-11", "2024-03-18",
        "2024-03-18", "2024-03-25", "2024-03-25", "2024-04-01", "2024-04-01"
    ],
    "speaker": [
        "Margaret Thornton", "James Whitfield", "Sarah Okonkwo", "David Hargreaves",
        "Emily Chen", "Robert MacLeod", "Fatima Hussain", "Thomas Brennan",
        "Catherine Lloyd", "Andrew Patel", "Helen Murray", "Kwame Asante",
        "Patricia Gallagher", "Ravi Sharma", "Fiona Blackwood", "Marcus Johnson",
        "Alison Crawford", "Yusuf Ali", "Diana Novak", "Christopher Reeves"
    ],
    "party": [
        "Labour", "Conservative", "Labour", "Conservative", "Labour",
        "SNP", "Labour", "Conservative", "Liberal Democrat", "Labour",
        "SNP", "Labour", "Conservative", "Labour", "SNP",
        "Conservative", "Labour", "Labour", "Liberal Democrat", "Conservative"
    ],
    "speech_text": [
        "The National Health Service remains the cornerstone of our social contract with the British people. We must invest in frontline services and address the chronic staffing shortages that are putting patients at risk. The waiting lists have grown to record levels and this government must take urgent action to reduce them.",
        "Economic growth is the foundation upon which all public services depend. By cutting taxes and reducing unnecessary regulation, we can unleash the potential of British businesses. The private sector, not the state, is the engine of prosperity and job creation in this country.",
        "The climate crisis demands immediate and ambitious action from this House. We cannot afford to delay investment in renewable energy infrastructure. Our children and grandchildren will judge us by the decisions we make today on carbon emissions and environmental protection.",
        "Immigration policy must be fair but firm. We need a points-based system that attracts the skilled workers our economy needs while maintaining control of our borders. The British people voted for sovereignty and we must deliver on that promise.",
        "Education is the great equaliser in our society. Every child, regardless of their background or postcode, deserves access to excellent teaching and modern facilities. We must close the attainment gap between the most and least disadvantaged pupils in our schools.",
        "Scotland's interests are consistently overlooked by this Westminster government. The devolution settlement is being undermined at every turn. The people of Scotland deserve the right to determine their own future and to have their voice heard on matters that affect their daily lives.",
        "The housing crisis is devastating communities across the country. Young people cannot afford to buy their first home and rents are consuming an ever larger share of household incomes. We need a massive programme of social housing construction to address this emergency.",
        "Defence spending must remain a top priority for this government. The threats we face from hostile state actors are growing more complex and more dangerous. We must ensure our armed forces have the equipment and resources they need to keep this nation safe.",
        "Mental health services are chronically underfunded and overstretched. Too many people are waiting months or even years for treatment they desperately need. Parity of esteem between physical and mental health must become a reality, not just a slogan.",
        "The cost of living crisis is hitting working families hardest. Energy bills have skyrocketed and food prices continue to rise at alarming rates. The government must do more to support those who are struggling to heat their homes and feed their children.",
        "The fishing communities of Scotland have been betrayed by broken promises on Brexit. Access to our waters and fair quota allocations are essential for the survival of these coastal communities. We will continue to fight for the rights of Scottish fishermen in this House.",
        "Public transport infrastructure is failing passengers across the north of England. Years of underinvestment have left communities isolated and reliant on expensive and unreliable services. We need a transport revolution that connects people to jobs, education, and opportunity.",
        "Law and order must be restored to our streets. Violent crime and antisocial behaviour are blighting communities and the police need the resources and powers to tackle these problems effectively. We will always stand on the side of victims and law-abiding citizens.",
        "The arts and creative industries contribute billions to our economy and enrich the cultural life of this nation. Cuts to arts funding have devastated regional theatres, museums, and music venues. Investment in culture is not a luxury; it is an investment in our communities and our identity.",
        "Rural communities in Scotland face unique challenges that this government fails to understand. Broadband connectivity, access to healthcare, and affordable transport are not luxuries but necessities. The centralisation of services in urban areas is leaving rural Scotland behind.",
        "Free trade agreements are essential for our post-Brexit economic strategy. British goods and services must have access to global markets on the most favourable terms possible. We are building new trading relationships that will benefit every region of the United Kingdom.",
        "Workers' rights must be strengthened, not eroded, in the modern economy. The rise of zero-hours contracts and the gig economy has created a new class of precarious employment. Every worker deserves fair pay, decent conditions, and the security of knowing their rights are protected by law.",
        "Child poverty is a stain on our national conscience. Over four million children in this country are growing up in poverty, and the numbers are rising. We must reform the welfare system to ensure that no child goes hungry and every family has a decent standard of living.",
        "Local government is the backbone of democracy in this country. Councils are being asked to do more with less, and essential services are being cut to the bone. Proper funding for local authorities is not just desirable; it is essential for the health of our democratic institutions.",
        "Science and innovation are the keys to our future prosperity. The United Kingdom has world-leading universities and research institutions that must be supported and celebrated. Government investment in research and development is an investment in the jobs and industries of tomorrow."
    ]
}

# Create a pandas DataFrame
df = pd.DataFrame(speeches_data)

print(f"Dataset shape: {df.shape[0]} speeches, {df.shape[1]} columns")
df.head()

We apply the preprocessing steps from Practical 1: tokenise, lowercase, remove punctuation, and remove stopwords. We define a simple preprocessing function that encapsulates these steps.

In [ ]:
def preprocess(text):
    """Tokenise, lowercase, remove punctuation/numbers, and remove stopwords."""
    tokens = word_tokenize(text)
    tokens = [t.lower() for t in tokens if t.isalpha()]
    stop_words = set(stopwords.words('english'))
    tokens = [t for t in tokens if t not in stop_words]
    return tokens

# Apply preprocessing to all speeches
df["tokens"] = df["speech_text"].apply(preprocess)

print(f"Example tokens (first speech): {df['tokens'].iloc[0][:10]}...")
print(f"Total tokens in first speech: {len(df['tokens'].iloc[0])}")

We also create a single combined list of all tokens in the corpus, which we will use for several analyses below.

In [ ]:
# Combine all tokens into a single list for corpus-level analysis
all_tokens = [token for tokens in df["tokens"] for token in tokens]

print(f"Total tokens in corpus: {len(all_tokens)}")
print(f"Unique tokens in corpus: {len(set(all_tokens))}")

## Word Frequency Analysis

The simplest form of text analysis is counting words. Word frequencies tell us which terms dominate the corpus — they are often the first thing we examine in any text analysis project.

We use Python's `Counter` to count the frequency of each token across the entire corpus.

In [ ]:
# Count word frequencies across the entire corpus
word_freq = Counter(all_tokens)

# Display the 20 most common words
print("Top 20 most frequent words in the corpus:")
for word, count in word_freq.most_common(20):
    print(f"  {word:20s} {count}")

A bar chart makes the distribution easier to interpret visually. Let's plot the top 20 words.

In [ ]:
# Plot the top 20 most frequent words
top_20 = word_freq.most_common(20)
words, counts = zip(*top_20)

plt.figure(figsize=(12, 6))
plt.bar(words, counts, color="steelblue")
plt.xlabel("Word")
plt.ylabel("Frequency")
plt.title("Top 20 Most Frequent Words in the Hansard Corpus")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

Word frequencies tell us which terms dominate the corpus, but the most frequent words are not always the most informative. Words like "must", "government", and "services" appear frequently because they are common in parliamentary discourse generally — they do not necessarily distinguish one speech from another. We will return to this issue when we compare vocabulary across groups later in the practical.

## Word Clouds

A word cloud is a popular visualisation that displays words in sizes proportional to their frequency. While they are intuitive and visually appealing, they should be used with care: the size of words can be misleading, and they provide no information about context or relationships between words.

Let's generate a word cloud for the entire corpus.

In [ ]:
# Join all tokens into a single string for the word cloud generator
corpus_text = " ".join(all_tokens)

# Generate the word cloud
wordcloud = WordCloud(
    width=800, height=400,
    background_color='white',
    max_words=100
).generate(corpus_text)

# Display the word cloud
plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud: All Parliamentary Speeches')
plt.show()

### Comparative Word Clouds

Word clouds become more analytically useful when we compare across groups. Let's split the corpus by party and generate side-by-side word clouds for **Conservative** and **Labour** speeches. This allows us to visually compare the vocabulary priorities of each party.

In [ ]:
# Split tokens by party
conservative_tokens = [t for tokens in df[df["party"] == "Conservative"]["tokens"] for t in tokens]
labour_tokens = [t for tokens in df[df["party"] == "Labour"]["tokens"] for t in tokens]

# Join tokens into strings
conservative_text = " ".join(conservative_tokens)
labour_text = " ".join(labour_tokens)

# Generate word clouds
wc_conservative = WordCloud(
    width=800, height=400, background_color='white', max_words=100,
    colormap='Reds'
).generate(conservative_text)

wc_labour = WordCloud(
    width=800, height=400, background_color='white', max_words=100,
    colormap='Blues'
).generate(labour_text)

# Display side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].imshow(wc_conservative, interpolation='bilinear')
axes[0].set_title('Conservative Speeches', fontsize=14)
axes[0].axis('off')

axes[1].imshow(wc_labour, interpolation='bilinear')
axes[1].set_title('Labour Speeches', fontsize=14)
axes[1].axis('off')

plt.suptitle('Comparative Word Clouds by Party', fontsize=16)
plt.tight_layout()
plt.show()

### Strengths and Limitations of Word Clouds

**Strengths:**
- Intuitive and visually engaging — useful for presentations and exploratory analysis.
- Quickly communicate the dominant vocabulary of a corpus.

**Limitations:**
- Word size can be misleading — small differences in frequency may appear as large visual differences.
- No information about context — we cannot tell *how* a word is used, only that it appears frequently.
- Placement is random — spatial position carries no meaning.
- Can give a false sense of analytical rigour when used alone.

Word clouds are best used as a complement to other methods, not as a standalone analytical tool.

## Keywords-in-Context (KWIC)

Word frequency tells us *how often* a term appears, but not *how* it is used. Keywords-in-Context (KWIC), also known as **concordancing**, shows us the surrounding words each time a target term appears. This is crucial for understanding meaning in context.

NLTK provides a convenient `.concordance()` method via the `Text` object. Let's create an `nltk.Text` object from our corpus tokens.

In [ ]:
# Create an NLTK Text object from all preprocessed tokens
nltk_text = Text(all_tokens)

print(f"NLTK Text object created with {len(nltk_text)} tokens")

Now let's use `.concordance()` to examine how key terms are used in context. We will look at a few politically salient terms.

In [ ]:
# Show concordance for "government"
print("=" * 70)
print("Concordance for 'government':")
print("=" * 70)
nltk_text.concordance("government", width=75, lines=10)

In [ ]:
# Show concordance for "services"
print("=" * 70)
print("Concordance for 'services':")
print("=" * 70)
nltk_text.concordance("services", width=75, lines=10)

In [ ]:
# Show concordance for "communities"
print("=" * 70)
print("Concordance for 'communities':")
print("=" * 70)
nltk_text.concordance("communities", width=75, lines=10)

KWIC reveals *how* terms are used, not just how often — crucial for understanding meaning in context. For example, "services" might appear in discussions of the NHS, public transport, or defence, and only KWIC reveals which sense is intended in each case.

### Programmatic Access with `.concordance_list()`

The `.concordance()` method prints results directly to the screen. If you need to work with concordance results programmatically — for example, to filter, count, or export them — use `.concordance_list()` instead, which returns a list of `ConcordanceLine` objects.

In [ ]:
# Get concordance results as a list for programmatic access
concordance_results = nltk_text.concordance_list("economy", width=75)

print(f"Found {len(concordance_results)} occurrences of 'economy'\n")

# Display each result with its left and right context
for i, entry in enumerate(concordance_results):
    left = " ".join(entry.left)
    right = " ".join(entry.right)
    print(f"{i+1}. ...{left} [{entry.query}] {right}...")

This programmatic access is useful when you want to build more complex analyses on top of concordancing — for example, counting how often a keyword appears in a particular context, or extracting concordance lines for export to a spreadsheet.

## Collocations and N-grams

Individual word frequencies tell us what terms are common, but they miss an important feature of language: words do not appear in isolation. **Collocations** are word pairs (or longer sequences) that occur together more than we would expect by chance. Identifying collocations helps us understand the *phrases* and *concepts* that characterise a corpus, not just isolated terms.

For example, "climate" and "crisis" appearing together frequently tells us something different from each word appearing independently.

### Extracting Bigrams Manually

The simplest approach to finding multi-word patterns is to extract **bigrams** — consecutive pairs of words — and count their frequencies.

In [ ]:
# Extract bigrams from the corpus tokens
bigrams = list(nltk.bigrams(all_tokens))

# Count bigram frequencies
bigram_freq = Counter(bigrams)

# Display the 20 most common bigrams
print("Top 20 most frequent bigrams:")
for bigram, count in bigram_freq.most_common(20):
    print(f"  {bigram[0]:15s} {bigram[1]:15s} {count}")

Raw bigram frequency is useful but can be dominated by pairs containing very common words. A more sophisticated approach is to use a statistical measure to identify word pairs that co-occur more often than we would expect by chance.

### Collocations Using PMI

**Pointwise Mutual Information (PMI)** is a statistical measure that compares how often two words appear together versus how often we would expect them to co-occur if they were independent. A high PMI score means the words co-occur much more frequently than chance would predict.

NLTK's `BigramCollocationFinder` makes this easy to compute. We apply a minimum frequency filter to remove rare, uninformative pairs.

In [ ]:
from nltk.metrics import BigramAssocMeasures

# Create a BigramCollocationFinder from the corpus tokens
finder = BigramCollocationFinder.from_words(all_tokens)

# Apply a frequency filter: only consider bigrams that appear at least 2 times
finder.apply_freq_filter(2)

# Get the top 20 collocations by PMI score
pmi_collocations = finder.nbest(BigramAssocMeasures.pmi, 20)

print("Top 20 collocations by PMI score:")
for w1, w2 in pmi_collocations:
    print(f"  {w1} {w2}")

Collocations are word pairs that occur together more than we would expect by chance. The PMI-based collocations above identify conceptually meaningful phrases such as political terms, policy areas, and rhetorical constructions that simple frequency counts might miss.

Note that PMI can sometimes favour very rare pairs. The frequency filter helps mitigate this, but it is always good practice to inspect your collocations and consider whether they are substantively meaningful.

## Comparing Vocabulary Across Groups

One of the most powerful applications of descriptive text analysis in the social sciences is comparing language use across groups. In our Hansard data, the natural grouping variable is **party**. Do different parties use different vocabularies? Are some terms more dominant in one party's speeches than another's?

Let's compare the speeches of the four parties in our corpus.

In [ ]:
# Check the distribution of speeches by party
print("Number of speeches by party:")
print(df["party"].value_counts())

### Top Terms by Party

Let's find the 10 most frequent terms for each party and display them side by side.

In [ ]:
# Compute top 10 terms for each party
parties = df["party"].unique()
top_terms_by_party = {}

for party in sorted(parties):
    party_tokens = [t for tokens in df[df["party"] == party]["tokens"] for t in tokens]
    party_freq = Counter(party_tokens)
    top_terms_by_party[party] = party_freq.most_common(10)

# Display as a formatted table
print(f"{'Rank':<6}", end="")
for party in sorted(parties):
    print(f"{party:<28s}", end="")
print()
print("-" * 118)

for i in range(10):
    print(f"{i+1:<6}", end="")
    for party in sorted(parties):
        word, count = top_terms_by_party[party][i]
        print(f"{word} ({count}){'':<16}", end="")
    print()

We can also visualise this comparison using grouped bar charts.

In [ ]:
# Create a subplot for each party showing their top 10 terms
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

colours = {"Conservative": "#0087DC", "Labour": "#DC241f", "Liberal Democrat": "#FDBB30", "SNP": "#FDF38E"}

for idx, party in enumerate(sorted(parties)):
    words_list, counts_list = zip(*top_terms_by_party[party])
    axes[idx].barh(range(len(words_list)), counts_list, color=colours.get(party, "grey"))
    axes[idx].set_yticks(range(len(words_list)))
    axes[idx].set_yticklabels(words_list)
    axes[idx].invert_yaxis()
    axes[idx].set_title(party, fontsize=13)
    axes[idx].set_xlabel("Frequency")

plt.suptitle("Top 10 Terms by Party", fontsize=15)
plt.tight_layout()
plt.show()

### Type-Token Ratio (TTR)

The **type-token ratio** (TTR) is a simple measure of vocabulary diversity. It is calculated as:

$$\text{TTR} = \frac{\text{Number of unique words (types)}}{\text{Total number of words (tokens)}}$$

A higher TTR indicates greater vocabulary diversity. Let's compare TTR across parties.

In [ ]:
# Calculate type-token ratio for each party
ttr_data = []

for party in sorted(parties):
    party_tokens = [t for tokens in df[df["party"] == party]["tokens"] for t in tokens]
    n_tokens = len(party_tokens)
    n_types = len(set(party_tokens))
    ttr = n_types / n_tokens if n_tokens > 0 else 0
    ttr_data.append({
        "Party": party,
        "Tokens": n_tokens,
        "Types (unique)": n_types,
        "TTR": round(ttr, 3)
    })

ttr_df = pd.DataFrame(ttr_data)
print("Vocabulary diversity by party:")
ttr_df

Note that TTR is sensitive to text length — shorter texts tend to have higher TTR values because there are fewer opportunities for words to repeat. When comparing groups, be mindful of differences in the total number of tokens. In our sample, the parties have different numbers of speeches (and therefore different total token counts), which affects the TTR comparison.

Despite this caveat, TTR provides a useful first approximation of vocabulary diversity and can flag interesting differences for further investigation.

## Exercise

Choose a research question you can explore with the tools introduced in this practical. For example:

- How do different parties talk about the NHS?
- What are the most distinctive terms used by each speaker?
- Are there notable collocations that differ between Conservative and Labour speeches?

Apply **at least two** of the methods above (word frequencies, word clouds, KWIC, collocations, or comparative analysis) to investigate your question.

Use the skeleton code below as a starting point. **Note**: You can use your own text data if you prefer.

In [ ]:
# State your research question
# Research question: 
# YOUR RESEARCH QUESTION HERE

In [ ]:
# Method 1: Apply your first analytical method
# For example, word frequency analysis or KWIC for a specific term

# YOUR CODE HERE

In [ ]:
# Method 2: Apply your second analytical method
# For example, comparative word clouds or collocations

# YOUR CODE HERE

In [ ]:
# Interpretation: What did you find? Write your interpretation as comments below.

# YOUR INTERPRETATION HERE

## Appendix: Exercise Solution

Below is a worked example investigating the question: **How do Conservative and Labour MPs talk about the economy?**

We apply two methods: KWIC analysis and comparative word frequency.

In [ ]:
# --- Solution: Method 1 — KWIC for 'economy' by party ---

# Create NLTK Text objects for each party
conservative_tokens_sol = [t for tokens in df[df["party"] == "Conservative"]["tokens"] for t in tokens]
labour_tokens_sol = [t for tokens in df[df["party"] == "Labour"]["tokens"] for t in tokens]

text_con = Text(conservative_tokens_sol)
text_lab = Text(labour_tokens_sol)

print("=" * 70)
print("Conservative speeches — concordance for 'economy':")
print("=" * 70)
text_con.concordance("economy", width=75, lines=10)

print("\n" + "=" * 70)
print("Labour speeches — concordance for 'economy':")
print("=" * 70)
text_lab.concordance("economy", width=75, lines=10)

In [ ]:
# --- Solution: Method 2 — Comparative word frequencies ---

# Get frequency distributions for both parties
con_freq = Counter(conservative_tokens_sol)
lab_freq = Counter(labour_tokens_sol)

# Create a comparison DataFrame
all_words_set = set(list(con_freq.keys()) + list(lab_freq.keys()))
comparison_data = []
for word in all_words_set:
    comparison_data.append({
        "word": word,
        "Conservative": con_freq.get(word, 0),
        "Labour": lab_freq.get(word, 0)
    })

comp_df = pd.DataFrame(comparison_data)
comp_df["difference"] = comp_df["Conservative"] - comp_df["Labour"]
comp_df = comp_df.sort_values("difference")

print("Words most skewed towards Labour (negative difference):")
print(comp_df.head(10).to_string(index=False))

print("\nWords most skewed towards Conservative (positive difference):")
print(comp_df.tail(10).to_string(index=False))

In [ ]:
# --- Solution: Visualise the most distinctive terms ---

# Combine the top and bottom terms
top_lab = comp_df.head(10)
top_con = comp_df.tail(10)
distinctive = pd.concat([top_lab, top_con])

# Plot
plt.figure(figsize=(10, 8))
colours_bar = ["#DC241f" if d < 0 else "#0087DC" for d in distinctive["difference"]]
plt.barh(distinctive["word"], distinctive["difference"], color=colours_bar)
plt.xlabel("Frequency Difference (Conservative - Labour)")
plt.title("Most Distinctive Words: Conservative vs Labour")
plt.axvline(x=0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

**Interpretation**: The KWIC analysis shows that Conservative MPs tend to discuss the "economy" in the context of growth, trade, and the private sector, while Labour MPs reference it in the context of workers' rights and the gig economy. The comparative frequency analysis reveals that Labour speeches are more likely to use terms related to social welfare (e.g., "poverty", "children", "rights"), while Conservative speeches emphasise terms related to trade, security, and sovereignty. These patterns align with well-documented differences in party priorities and framing strategies.

## Bibliography

- Grimmer, J., Roberts, M. E., & Stewart, B. M. (2022). *Text as Data: A New Framework for Machine Learning and the Social Sciences*. Princeton University Press.
- Manning, C. D., Raghavan, P., & Schutze, H. (2008). *Introduction to Information Retrieval*. Cambridge University Press.
- Bird, S., Klein, E., & Loper, E. (2009). *Natural Language Processing with Python*. O'Reilly Media.
- Sinclair, J. (1991). *Corpus, Concordance, Collocation*. Oxford University Press.
- Baker, P. (2006). *Using Corpora in Discourse Analysis*. Continuum.
- Church, K. W., & Hanks, P. (1990). Word association norms, mutual information, and lexicography. *Computational Linguistics*, 16(1), 22-29.

---

**END OF FILE**